# $l_0$-constrained Ridge Regression

In this notebook, we perform cuts-generation experiments on $\ell_0$-constrained Ridge Regression, namely:
$$\underset{\|\boldsymbol{\beta}\|_0 \leq k}{\operatorname{min}} \frac{1}{n} \|\boldsymbol{X} \boldsymbol{\beta} - \boldsymbol{Y}\|_2^2 + \gamma \|\boldsymbol{\beta} \|_2^2$$
<!-- where we use $\tilde{\boldsymbol{\beta}}$ to denote the optimal solution.  -->

<!-- The data generation follows the procedure in [Bertsimas et al. (2019)](https://arxiv.org/pdf/1902.06547). Below are some quick links to different sections: 
- [Playground](#playground)
    - [Ground-truth Checking](#ground_truth_checking)
- [Formal Testing](#formal_testing)
- [Plotting](#plotting)
    - [Solved Instances](#sol_instance)
    - [Fixed Indices](#indices_fixed) -->

In [2]:
# --- import packages ---
import sys 
sys.path.append("../..") 
from source import utils,env,main,test
import numpy as np
import pandas as pd
import random
import math
import time

----

 ## <a id='playground'></a> Playground
 In this section, we use some synthetic instances to test the performance of two different cuts-generation methods.

In [8]:
# --- Generate instances for testing ---
random_seed = None
num = 50
dim = 500
sparsity_ = 10
rho_ = 0.5
SNR_ = 3.5
(X,Y,beta_true) = env.sparse_generation(num = num,
                              dim = dim,
                              sparsity=sparsity_,
                              rho = rho_,
                              SNR = SNR_,
                              random_seed = random_seed)

# gamma_ = (1 / (env.generate_gamma(X,sparsity_,3))) / num
gamma_ = 1.4

In [4]:
SSR_disp = True
SCG_disp = True

procedure = main.ridge_screen(
covariate=X,response=Y,gamma = gamma_,sparsity = sparsity_)

procedure.get_feas_beta_ub(relax_solver='gurobi')

if SSR_disp:
    # --- SSR ---
    print('=== SSR ===')

    current_time = time.time()
    S_SSR,Z_SSR = procedure.safe_screen()
    end_time = time.time()

    SSR_screen_time = end_time - current_time
    print(SSR_screen_time)
    print('support index:')
    print(procedure.support_index)
    print('zero index:')
    print(len(procedure.zero_index),procedure.zero_index)
    # print(procedure.zero_index)

if SCG_disp:
    # --- SCG ---
    print('=== SCG ===')
    current_time = time.time()
    S_inc,Z_inc = procedure.inclusive_cuts(max_num = sparsity_,max_len = 2)
    end_time = time.time()
    inc_find_time = end_time - current_time

    current_time = time.time()
    S_exc,Z_exc = procedure.exclusive_cuts(max_num = dim - sparsity_,max_len = 3)
    end_time = time.time()
    exc_find_time = end_time - current_time
    
    print('inclusive cuts:')
    print(inc_find_time)
    for i in Z_inc:
        print(i)

    
    print('exclusive cuts:')
    print(exc_find_time)
    print(len(S_exc))
    for i in S_exc:
        print(i)

In [6]:
# --- Model testing ---
org_test = False
SSR_test = True
SCG_test = False

time_lim = 15 * 60
output_log = 1
ite_lim = None
gap_lim = 0.01

if org_test == True:
    print('=== start solving original problem ===')
    current_time = time.time()
    org_model = utils.ridge_train(X = X,Y = Y,
                                  sparsity = sparsity_,gamma = gamma_,
                                  Outputlog= output_log,mip_gap = gap_lim,
                                  max_ite = ite_lim,
                                  max_min= time_lim)
    end_time = time.time()
    z_org = [var.X for var in org_model.getVars() if 'z' in var.VarName]
    support_org = [i for i in range(dim) if z_org[i] == 1]
    print(support_org)
    print('=== done ===')

if SSR_test == True:
    print('=== start solving with SSR ===')
    current_time = time.time()
    SSR_model = utils.ridge_train(X = X,Y = Y,
                              sparsity=sparsity_,gamma = gamma_,
                              Outputlog=output_log,mip_gap = gap_lim,
                              S_set = S_SSR,Z_set = Z_SSR,
                              max_ite = ite_lim,
                              max_min= time_lim)
    end_time = time.time()
    z_SSR = [var.X for var in SSR_model.getVars() if 'z' in var.VarName]
    support_SSR = [i for i in range(dim) if z_SSR[i] == 1]
    print(support_SSR)
    print('total time:',SSR_screen_time + end_time - current_time)
    print('=== done ===')
    
if SCG_test == True:
    print('=== start solving with SCG ===')
    current_time = time.time()
    SCG_model = utils.ridge_train(X = X,Y = Y,
                              sparsity=sparsity_,gamma = gamma_,
                              Outputlog=output_log,mip_gap = gap_lim,
                              S_set = S_inc+S_exc ,Z_set = Z_inc+Z_exc,
                              max_ite = ite_lim,max_min = time_lim)
    end_time = time.time()
    z_SCG = [var.X for var in SCG_model.getVars() if 'z' in var.VarName]
    support_SCG = [i for i in range(dim) if z_SCG[i] == 1]
    print(support_SCG)
    print('total time:',inc_find_time + exc_find_time + end_time - current_time)
    print('=== done ===')